# Deep Agents - Virtual File System (Gemini Version)
This notebook demonstrates how to give an agent a virtual file system to prevent context window overload. Fully configured for **Gemini** and ready for the Manager's Demo.

In [12]:
# 1. Install required packages (Added python-dotenv)
#!pip install -qU langgraph langchain-google-genai pydantic typing_extensions python-dotenv

import os
from dotenv import load_dotenv

# 2. Load API key from .env file
print("🔄 Loading API key from .env file...")
load_dotenv()  
# 3. Verify if the key was loaded successfully
if "GOOGLE_API_KEY" not in os.environ or not os.environ["GOOGLE_API_KEY"]:
    print("❌ ERROR: GOOGLE_API_KEY not found!")
    print("Please check if your .env file exists and has the key in this format: GOOGLE_API_KEY=your_key")
else:
    print("✅ API Key successfully loaded from .env! Move to the next cell.")

🔄 Loading API key from .env file...
✅ API Key successfully loaded from .env! Move to the next cell.


In [15]:
from typing import Annotated, Dict
from typing_extensions import TypedDict
from langchain_core.messages import ToolMessage, BaseMessage
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.prebuilt import InjectedState, create_react_agent
from langgraph.types import Command
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Reducer Function to merge files in State (Prevents in-place mutation bugs)
def file_reducer(left: dict | None, right: dict | None) -> dict:
    if left is None:
        return right or {}
    if right is None:
        return left or {}
    return {**left, **right}

# 2. Define Agent State using the Reducer (V1.0+ Compatible)
class DeepAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    files: Annotated[Dict[str, str], file_reducer]  # <--- Reducer added here
    remaining_steps: int  
    is_last_step: bool    

# 3. Define File Tools with Defensive Defaults to bypass Pydantic bugs
@tool
def ls(state: Annotated[dict, InjectedState] = None) -> list[str]:
    """List all files in the virtual filesystem. Use this to check what files currently exist."""
    if state is None:
        return []
    return list(state.get("files", {}).keys())

@tool
def read_file(file_path: str, state: Annotated[dict, InjectedState] = None) -> str:
    """Read content from a file in the virtual filesystem."""
    if state is None:
        return "Error: No state injected"
    files = state.get("files", {})
    if file_path not in files:
        return f"Error: File '{file_path}' not found"
    return files[file_path]

@tool
def write_file(
    file_path: str, 
    content: str, 
    tool_call_id: Annotated[str, InjectedToolCallId] = None  # <--- Bypasses state injection issues completely
) -> Command:
    """Write content to a file in the virtual filesystem."""
    return Command(
        update={
            # Reducer will automatically merge this file with the existing ones!
            "files": {file_path: content},
            "messages": [ToolMessage(f"Successfully saved content to {file_path}", tool_call_id=tool_call_id)],
        }
    )

# 4. Initialize Gemini Model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 5. System Prompt
system_prompt = """You are an advanced AI assistant with access to a virtual file system.
When given a task with lots of information:
1. Use `write_file` to save data into files instead of just printing it.
2. Use `ls` to check what files exist.
3. Use `read_file` to read data back when needed.
Always keep the user updated on what files you are creating."""

# 6. Compile the LangGraph Agent
agent = create_react_agent(
    model=llm,
    tools=[ls, read_file, write_file],
    state_schema=DeepAgentState,
    prompt=system_prompt
)

print("✅ File System Agent successfully created with Gemini! Ready for tasks.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ File System Agent successfully created with Gemini! Ready for tasks.


C:\Users\ARYAN\AppData\Local\Temp\ipykernel_19228\89801624.py:70: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [16]:
# =====================================================================
#test:
# =====================================================================

# Default Task: Create, List, and Read files
user_input = (
    "Step 1: Create a file named 'marketing_ideas.txt' and write 3 marketing strategies for a new AI product in it. "
    "Step 2: Use the 'ls' tool to verify the file was created. "
    "Step 3: Read the file, and create a new file called 'summary.txt' that contains a 1-line summary of those strategies."
)

print(f"🚀 Running Agent for task:\n{user_input}\n")
print("=" * 60)

inputs = {
    "messages": [("user", user_input)],
    "files": {} 
}

final_event = None
for event in agent.stream(inputs, stream_mode="values"):
    message = event["messages"][-1]
    message.pretty_print()
    final_event = event

print("\n" + "=" * 60)
print("📁 FINAL VIRTUAL FILESYSTEM STATE :")
print("=" * 60)

if final_event and "files" in final_event:
    for filename, content in final_event["files"].items():
        print(f"\n📄 File: {filename}")
        print("-" * 30)
        print(content)
        print("-" * 30)

🚀 Running Agent for task:
Step 1: Create a file named 'marketing_ideas.txt' and write 3 marketing strategies for a new AI product in it. Step 2: Use the 'ls' tool to verify the file was created. Step 3: Read the file, and create a new file called 'summary.txt' that contains a 1-line summary of those strategies.

================================ Human Message =================================

Step 1: Create a file named 'marketing_ideas.txt' and write 3 marketing strategies for a new AI product in it. Step 2: Use the 'ls' tool to verify the file was created. Step 3: Read the file, and create a new file called 'summary.txt' that contains a 1-line summary of those strategies.
================================== Ai Message ==================================

[{'type': 'text', 'text': "I'm creating a file named 'marketing_ideas.txt' with three marketing strategies for a new AI product.", 'extras': {'signature': 'CqEHAQw51sfBnlSlfTQVmFHTEJw0HJdb5yJJd9j9/jhMaC83k2DIcC0ADwEDkMBU9qmWaqBc/sPBcWt

In [18]:
final_event


{'messages': [HumanMessage(content="Step 1: Create a file named 'marketing_ideas.txt' and write 3 marketing strategies for a new AI product in it. Step 2: Use the 'ls' tool to verify the file was created. Step 3: Read the file, and create a new file called 'summary.txt' that contains a 1-line summary of those strategies.", additional_kwargs={}, response_metadata={}, id='64915543-cefa-44d3-ab2e-16b119d022c8'),
  AIMessage(content=[{'type': 'text', 'text': "I'm creating a file named 'marketing_ideas.txt' with three marketing strategies for a new AI product.", 'extras': {'signature': 'CqEHAQw51sfBnlSlfTQVmFHTEJw0HJdb5yJJd9j9/jhMaC83k2DIcC0ADwEDkMBU9qmWaqBc/sPBcWtdMtXB2oM1i+WfeLeciXrBeMTwi4L7y/0yescQxaEvzd1PijUHyCxgwUpnt1mipxsoVezcvFBa6w8Dc66mfYTRz7nq7ts8RLqf9bn2PZnwNSRmXkoWBMtkHzNrWx0k3UTKUK9mjpbcYUneJuQktAzIG/01oI5d7NEBboz/HTT1oyOxG49n0R2DxX3XGevJ5lScND4SMJTOGgkNy7aOGGr2rPbHGVUL1BQmUo6Dmfyj8l2cTdOgKNeowhf08yasgfH0xi2rqpzuSaOo36G+YmxrRknjP60sd4VzFPPR9SURzxq2Nqsv84N9IoQfYxWZoTYezLmdfzV7NUl

In [17]:
final_event["files"].items()

dict_items([('marketing_ideas.txt', "1. Focus on problem-solving: Highlight how the AI product solves specific pain points for target users.\n2. Emphasize ease of integration: Showcase the product's compatibility and seamless integration with existing systems.\n3. Offer a freemium model: Provide a basic version for free to attract users and then upsell premium features."), ('summary.txt', 'The marketing strategies for the new AI product focus on problem-solving, seamless integration, and a freemium model.')])